# Tracetools Tests

This notebook tests the dialoghelper **tracetools** module, which provides LLM-accessible function tracing using Python 3.12's `sys.monitoring`.

## Key Functions

- `tracetool(sym, args, kwargs, target_func)` — Trace execution of a callable, returning per-call stack traces and variable snapshots
- `fmt_trace(traces)` — Format raw trace output as readable markdown tables

**Run each cell in order.**

In [ ]:
# Cell 1: Imports
from dialoghelper.tracetools import tracetool, fmt_trace
from IPython.display import Markdown, display

print('Tracetools imports ready!')

In [ ]:
# Cell 2: Define a simple function to trace
def demo(n, m='x'):
    total = 0
    for i in range(n): total += i
    return m * total

# Quick test
print(f'demo(5) = {demo(5)!r}')
print(f'demo(5, m="y") = {demo(5, m="y")!r}')

In [ ]:
# Cell 3: Trace the demo function (raw output)
# tracetool returns a list of (stack_str, trace_dict) tuples
r = tracetool(sym='demo', args=[5], kwargs={'m': 'y'})
print(f'Got {len(r)} trace(s)')
print(f'Type: {type(r)}')
print()
# Show raw structure
for i, (stack, trace) in enumerate(r):
    print(f'Trace {i}:')
    print(f'  Stack: {stack!r}')
    for src, (hits, vars) in trace.items():
        print(f'  {src!r}: hits={hits}, vars={vars}')

In [ ]:
# Cell 4: Formatted trace (rendered as markdown tables)
# fmt_trace returns markdown; Markdown() renders it inline
Markdown(fmt_trace(r))

In [ ]:
# Cell 5: Alternative display via display()
# This also works - goes through IPython's display publisher
r2 = tracetool(sym='demo', args=[3])
display(Markdown(fmt_trace(r2)))

In [ ]:
# Cell 6: Trace a stdlib function
import textwrap

def quotefunc(s):
    return textwrap.wrap('aaa ' * 10, width=10, subsequent_indent='> ')

# Trace quotefunc but target the internal _wrap_chunks method
r3 = tracetool(
    sym='quotefunc',
    args=['test'],
    target_func='textwrap.TextWrapper._wrap_chunks'
)
print(f'Got {len(r3)} trace(s) from _wrap_chunks')
Markdown(fmt_trace(r3))

In [ ]:
# Cell 7: Trace a recursive function
def fib(n):
    if n <= 1: return n
    return fib(n - 1) + fib(n - 2)

r4 = tracetool(sym='fib', args=[4])
print(f'Got {len(r4)} trace(s) — one per recursive call')
Markdown(fmt_trace(r4))

## Summary

If all cells ran successfully, you've verified:

- **tracetool()** — Traces function execution with variable snapshots (raw data)
- **fmt_trace()** — Formats traces as markdown tables (Source | Hits | Variables)
- **Markdown rendering** — `Markdown(fmt_trace(r))` renders tables inline in Dialeng
- **target_func** — Can trace internal functions called by the main callable
- **Recursive tracing** — Each recursive call gets its own trace entry

### How It Works

```
tracetool(sym='func_name', args=[...], kwargs={...})
  → resolve(sym)           # Get callable from dotted path
  → tracefunc(callable)    # sys.monitoring traces execution
  → [(stack, trace_dict)]  # Per-call: stack trace + line-by-line variable snapshots

fmt_trace(traces)
  → Markdown table with Source, Hits, Variables columns
  → Variables show type+repr; changed vars show evolution with →
```

### Requirements

- Python 3.12+ (for `sys.monitoring`)
- `tracefunc` package (installed as dependency)
- `toolslm` package (for `resolve()` function)